# Akita — Predicting 3D genome folding from DNA sequence (Colab inference)

**Paper:** Fudenberg, Kelley & Pollard, *Nature Methods* 2020 — [Predicting 3D genome folding from DNA sequence with Akita](https://www.nature.com/articles/s41592-020-0958-x)
**Code:** [calico/basenji → manuscripts/akita](https://github.com/calico/basenji/tree/master/manuscripts/akita)

This notebook runs **inference only** with the authors' pre-trained model: give it a
1 Mb (2^20 bp) DNA region from hg38 and it predicts a Hi-C / Micro-C **contact map**
(log observed/expected) for that locus — the visual "genome folding" result from the paper.

### How to use
1. Open in **Google Colab** (Runtime > CPU is fine — no GPU needed for one prediction).
2. Run the cells top to bottom.
3. The predicted contact map is saved as `akita_pred.png` at the end. Download it and
   commit it into `papers/04-akita/` in the repo.

> **Heads-up on dependencies.** Basenji pins older library versions, so on Colab's default
> environment you may hit version conflicts. If an `import` fails right after the install
> cell: **Runtime > Restart session**, then re-run from the *Imports* cell (skip install).
> A little trial-and-error the first time is normal with these genomics tools.

## 1. Install Akita (basenji) + dependencies

In [ ]:
# Takes a few minutes. If imports later fail: Runtime > Restart session,
# then skip this cell and continue from the Imports cell.
!git clone --depth 1 https://github.com/calico/basenji.git /content/basenji
!pip install -q /content/basenji
!pip install -q cooltools pysam
print("install step done")

## 2. Download the pre-trained model

In [ ]:
import os
os.chdir('/content/basenji/manuscripts/akita')   # params.json + data/ live here
# Trained weights: human Hi-C + Micro-C, hg38, 2048 bp resolution
if not os.path.isfile('model_best.h5'):
    !wget -q https://storage.googleapis.com/basenji_hic/1m/models/9-14/model_best.h5
print('model present:', os.path.isfile('model_best.h5'), '| cwd:', os.getcwd())

## 3. Imports + load the model

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pysam
os.environ["CUDA_VISIBLE_DEVICES"] = '-1'   # run on CPU (robust; avoids GPU/cuDNN issues)
import tensorflow as tf
from cooltools.lib.numutils import set_diag
from basenji import dna_io, seqnn

with open('params.json') as f:
    params = json.load(f)
seqnn_model = seqnn.SeqNN(params['model'])
seqnn_model.restore('model_best.h5')
print('model loaded successfully')

## 4. Target cell types + reshape constants

In [ ]:
hic_targets = pd.read_csv('data/targets.txt', sep='\t')
hic_num_to_name = dict(zip(hic_targets['index'].values, hic_targets['identifier'].values))

with open('data/statistics.json') as f:
    data_stats = json.load(f)
seq_length     = data_stats['seq_length']            # 2^20 = 1,048,576 bp input
hic_diags      = data_stats['diagonal_offset']
target_crop    = data_stats['crop_bp'] // data_stats['pool_width']
target_length1 = data_stats['seq_length'] // data_stats['pool_width']
target_length1_cropped = target_length1 - 2 * target_crop
print('input seq length:', seq_length)
print('cell types:', list(hic_num_to_name.values()))

## 5. Helper: flattened vector -> symmetric contact matrix

In [ ]:
def from_upper_triu(vector_repr, matrix_len, num_diags):
    z = np.zeros((matrix_len, matrix_len))
    triu_tup = np.triu_indices(matrix_len, num_diags)
    z[triu_tup] = vector_repr
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)
    return z + z.T

## 6. Get the hg38 sequence source (~1 GB, one-time download)

In [ ]:
if not os.path.isfile('data/hg38.ml.fa'):
    print('downloading hg38 fasta (one-time, ~1 GB)...')
    !curl -o data/hg38.ml.fa.gz https://storage.googleapis.com/basenji_barnyard2/hg38.ml.fa.gz
    !gunzip data/hg38.ml.fa.gz
fasta_open = pysam.Fastafile('data/hg38.ml.fa')
print('fasta ready')

## 7. Pick a locus and one-hot encode it
The region **must span exactly 2^20 bp (1,048,576)**. Change `region` to fold a different locus.

In [ ]:
region = 'chr12:115163136-116211712'   # example locus from the paper (chr12)

chrm, coords = region.split(':')
seq_start, seq_end = (int(x) for x in coords.split('-'))
assert seq_end - seq_start == seq_length, f'region must span exactly {seq_length} bp'

seq = fasta_open.fetch(chrm, seq_start, seq_end).upper()
seq_1hot = dna_io.dna_1hot(seq)          # shape [1048576, 4]
print('one-hot sequence shape:', seq_1hot.shape)

## 8. Predict

In [ ]:
# model expects a batch: [n_regions, 2^20, 4]
pred = seqnn_model.model.predict(np.expand_dims(seq_1hot, 0))
print('prediction shape:', pred.shape)   # [1, #pixels, #cell types]

## 9. Plot + save the predicted contact map

In [ ]:
target_index = 0        # 0 = first cell type (see list printed in step 4)
vmin, vmax = -2, 2

mat = from_upper_triu(pred[:, :, target_index], target_length1_cropped, hic_diags)

plt.figure(figsize=(6, 6))
im = plt.matshow(mat, fignum=False, cmap='RdBu_r', vmin=vmin, vmax=vmax)
plt.colorbar(im, fraction=.04, pad=0.05, ticks=[-2, -1, 0, 1, 2])
plt.title(f'Akita prediction - {hic_num_to_name[target_index]}\n{region}', y=1.08)
plt.savefig('akita_pred.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved akita_pred.png  -> download it and commit into papers/04-akita/')